In [1]:
!pip install pandas beautifulsoup4 scikit-learn textstat nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 239.2/239.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 77.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
from bs4 import BeautifulSoup
import textstat
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np
import re

# Download NLTK stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
# Load the dataset
# --- CHANGE 'your_dataset.csv' TO THE ACTUAL FILENAME ---
# --- This assumes the dataset is unzipped in the same folder ---
try:
    df = pd.read_csv("data (1).csv")
except FileNotFoundError:
    print("ERROR: Make sure the dataset file is unzipped and the filename is correct.")
    # Create a dummy df to avoid crashing the rest of the script
    df = pd.DataFrame({'html_content': ['<html><body><p>Hello world</p></body></html>']})

# --- CHANGE 'html_content' IF YOUR COLUMN NAME IS DIFFERENT ---
HTML_COLUMN = 'html_content'

# 1. Function to Parse HTML and Extract Features
def process_html(html_content):
    if not isinstance(html_content, str):
        return "", 0, 0 # Return empty values for bad data (e.g., NaN)

    # 1a. Parse HTML
    soup = BeautifulSoup(html_content, 'html.parser')

    # Get all text
    text = soup.get_text(separator=' ', strip=True)

    # 1b. Clean text for NLP
    text = re.sub(r'\s+', ' ', text) # Remove extra whitespace
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower() # Remove punctuation/numbers

    # 1c. Engineer Features
    try:
        # Feature 1: Readability Score (Flesch-Kincaid Grade)
        readability = textstat.flesch_kincaid_grade(text)
    except:
        readability = 100 # Assign a high (bad) score if textstat fails

    # Feature 2: Word Count
    word_count = len(text.split())

    return text, readability, word_count

# 2. Apply the function to your DataFrame
print("Processing HTML and engineering features...")
df[['cleaned_text', 'readability', 'word_count']] = df[HTML_COLUMN].apply(
    lambda x: pd.Series(process_html(x))
)

# 3. Drop rows that were empty or failed parsing
df = df[df['word_count'] > 20].reset_index(drop=True)

print("Processing complete.")
print(df[['cleaned_text', 'readability', 'word_count']].head())

Processing HTML and engineering features...
Processing complete.
                                        cleaned_text  readability  word_count
0  cyber security blog img height width styledisp...   962.788532        2452
1  top  cybersecurity awareness tips how to stay ...   962.591223        2452
2   cyber defense tips to stay secure at work and...   583.891563        1484
3  cybersecurity best practices  cybersecurity an...   461.158497        1158
4  network security  understanding the basics pla...  1126.916750        2868


In [4]:
print("Starting duplicate detection...")

# 1. Initialize TF-IDF Vectorizer
# We limit features to 2000 to make it run fast
vectorizer = TfidfVectorizer(stop_words='english', max_features=2000)

# 2. Create the TF-IDF matrix from our cleaned text
try:
    tfidf_matrix = vectorizer.fit_transform(df['cleaned_text'])
except ValueError:
    print("Not enough data to run detection. Skipping.")
    tfidf_matrix = None

if tfidf_matrix is not None:
    # 3. Calculate Cosine Similarity
    # This compares every document to every other document
    print("Calculating cosine similarity matrix...")
    cosine_sim_matrix = cosine_similarity(tfidf_matrix)

    # 4. Find and display duplicate pairs
    # We look for pairs with similarity > 0.9 (i.e., 90% similar)
    threshold = 0.90
    duplicates = np.where(cosine_sim_matrix > threshold)

    # Collect unique pairs
    duplicate_pairs = set()
    for i, j in zip(duplicates[0], duplicates[1]):
        if i < j: # Avoid self-comparison and duplicate pairs (A,B) vs (B,A)
            duplicate_pairs.add((i, j))

    print(f"\n--- Found {len(duplicate_pairs)} Near-Duplicate Pairs (Threshold > {threshold*100}%) ---")

    # Display the top 5 pairs
    for i, (doc1_index, doc2_index) in enumerate(list(duplicate_pairs)[:5]):
        similarity = cosine_sim_matrix[doc1_index, doc2_index]
        print(f"\nPair {i+1}: Doc {doc1_index} and Doc {doc2_index} (Similarity: {similarity:.2f})")
        print(f"Doc {doc1_index} text: {df.loc[doc1_index, 'cleaned_text'][:100]}...")
        print(f"Doc {doc2_index} text: {df.loc[doc2_index, 'cleaned_text'][:100]}...")
else:
    print("Skipped duplicate detection due to lack of data.")

Starting duplicate detection...
Calculating cosine similarity matrix...

--- Found 0 Near-Duplicate Pairs (Threshold > 90.0%) ---


In [12]:
import joblib # Make sure joblib is imported

print("\n--- Building Content Quality Classification Model ---")

# 1. Define the Quality Rule
# The data is too small for a robust ML model, so we will use a
# "Rule-Based Model" derived from our data. This is our "model".
word_count_median = df['word_count'].median()
print(f"Using Rule-Based Model. 'Good' Quality = Word Count > {word_count_median:.1f}")

# 2. Define the "model" as a simple function
def predict_quality(readability, word_count):
    # Our "model" is the business rule we defined
    if word_count > 2452.0:
        return "Good"
    else:
        return "Bad"

# 3. Show a "real-time" analysis demo
print("\n--- Demo: Predicting on new data ---")

demo_data = pd.DataFrame({
    'readability': [950, 800, 1000],  # Readability doesn't affect this rule
    'word_count': [3000, 150, 2000]   # This is what the rule checks
})
# Expected: Good (3000 > 2452), Bad (150 < 2452), Bad (2000 < 2452)

# Apply our rule-based model
demo_data['PREDICTED_QUALITY'] = demo_data.apply(
    lambda row: predict_quality(row['readability'], row['word_count']), axis=1
)

print(demo_data)

# 4. Save only the duplicate detection models
# We don't need to save a quality model, since it's just a function
# that we will copy to our Streamlit app.
print("\nSaving duplicate detection models for Streamlit...")
if 'vectorizer' in locals():
    joblib.dump(vectorizer, 'tfidf_vectorizer.joblib')
    print("Saved 'tfidf_vectorizer.joblib'")
else:
    print("ERROR: 'vectorizer' not found. Run Step 2.")

if 'tfidf_matrix' in locals():
    joblib.dump(tfidf_matrix, 'tfidf_matrix.joblib')
    print("Saved 'tfidf_matrix.joblib'")
else:
    print("ERROR: 'tfidf_matrix' not found. Run Step 2.")


print("\n\n--- 20-MINUTE BLITZ COMPLETE ---")


--- Building Content Quality Classification Model ---
Using Rule-Based Model. 'Good' Quality = Word Count > 2452.0

--- Demo: Predicting on new data ---
   readability  word_count PREDICTED_QUALITY
0          950        3000              Good
1          800         150               Bad
2         1000        2000               Bad

Saving duplicate detection models for Streamlit...
Saved 'tfidf_vectorizer.joblib'
Saved 'tfidf_matrix.joblib'


--- 20-MINUTE BLITZ COMPLETE ---
